# 24 VLM Mechanics Baseline

Purpose: create a VLM input manifest and, after captions/tags/scores are filled, train an event-level baseline from VLM mechanics features.

Runtime: CPU for template/baseline. The actual VLM captioning step is run in `24b_hf_vlm_captioning.ipynb` when using an open-source HF VLM, or can be filled externally.

Outputs:
- `features/{vlm_feature_id}/manifest.parquet`
- `predictions/{vlm_run_id}/predictions_v1.parquet`
- `predictions/{vlm_run_id}/metrics_v1.json`
- `reports/preflight/vlm_feature_baseline_{vlm_run_id}.json`

Next: `25_method_evaluation.ipynb` to compare this method against pose/object TCN and raw video methods.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください.\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は notebook の最初の cell より前に次を実行してください.\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, resolve_statcast_date_range, run_id, stage_settings

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
VLM_RUN_ID = run_id(RUN_PROFILE, 'vlm_run_id', 'vlm_mechanics_mlb_2024_2026_v2')
VLM_FEATURE_ID = artifact_id(RUN_PROFILE, 'vlm_feature_id')
VLM_SETTINGS = stage_settings(RUN_PROFILE, 'vlm_mechanics')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('VLM_RUN_ID =', VLM_RUN_ID)
print('VLM_FEATURE_ID =', VLM_FEATURE_ID)


In [ ]:
import importlib
import sport_pipeline.models.vlm as vlm_module
import sport_pipeline.models.vlm.feature_baseline as vlm_feature_baseline_module
importlib.reload(vlm_feature_baseline_module)
importlib.reload(vlm_module)
from sport_pipeline.models.vlm import build_vlm_feature_template

MAX_CLIPS = VLM_SETTINGS.get('max_clips', 500)
FORCE_REBUILD_TEMPLATE = os.environ.get('FORCE_REBUILD_VLM_TEMPLATE', '0').lower() in {'1', 'true', 'yes'}
vlm_manifest = BASE_DIR / f'features/{VLM_FEATURE_ID}/manifest.parquet'

if FORCE_REBUILD_TEMPLATE or not vlm_manifest.exists():
    template_outputs = build_vlm_feature_template(
        BASE_DIR,
        clip_run_id=FULL_RUN_ID,
        vlm_feature_id=VLM_FEATURE_ID,
        max_clips=MAX_CLIPS,
    )
    for name, path in template_outputs.items():
        print(name, '->', path, 'exists=', path.exists())
else:
    template_outputs = {'template': vlm_manifest}
    print('existing VLM manifest kept ->', vlm_manifest)

print('If vlm_caption / vlm_labels / numeric_features are empty, run notebooks/24b_hf_vlm_captioning.ipynb before RUN_BASELINE=True.')


In [ ]:
# Set this True only after the VLM manifest has captions/tags/scores.
baseline_env = os.environ.get('RUN_VLM_BASELINE')
RUN_BASELINE = (
    baseline_env.lower() in {'1', 'true', 'yes'}
    if baseline_env is not None
    else bool(VLM_SETTINGS.get('run_baseline_default', VLM_SETTINGS.get('execute_default', False)))
)
print('RUN_BASELINE =', RUN_BASELINE)

import importlib
import sport_pipeline.models.vlm as vlm_module
import sport_pipeline.models.vlm.feature_baseline as vlm_feature_baseline_module
importlib.reload(vlm_feature_baseline_module)
importlib.reload(vlm_module)
from sport_pipeline.models.vlm import run_vlm_feature_baseline

if RUN_BASELINE:
    outputs = run_vlm_feature_baseline(
        BASE_DIR,
        prediction_run_id=VLM_RUN_ID,
        vlm_feature_id=VLM_FEATURE_ID,
        require_non_empty=bool(VLM_SETTINGS.get('require_non_empty', True)),
    )
    for name, path in outputs.items():
        print(name, '->', path, 'exists=', path.exists())
else:
    print('RUN_BASELINE=False: template was created, but no VLM predictions were written.')
    print('NEXT after filling VLM features: rerun this cell with RUN_BASELINE=True, then run notebooks/25_method_evaluation.ipynb')
